# 005-E3 ICEBERG NP250 (DEV run, not submission)

- 250 molecules from `np250_gl_input.json` (= `np250_shortlists_top50.json` top50, 0 mismatch verified locally) truncated to **top-25** (first 25, order preserved) — same window / same SMILES as the 30-mol precedent (`iceberg-compare-30`) and e3 top25 caliber. Full top50-vs-top25 fusion comparison is done offline after retrieval.
- Weights: `ahmedberatozer/casmi26-iceberg` = MassSpecGym **msg_all** mirror (`ckpt/gen.pt` 14561244B + `ckpt/inten.pt` 13793037B); verified size/sha256 vs dataset MANIFEST in-run. Upstream official ms-pred README: msg_all Dropbox `rlkey=f1n6pbzx94g1k2el2wcbmee61` (preferred), msg_simulation is the other folder. **NIST weights excluded everywhere.**
- Chain: reuse iceberg-compare-30 (PyPI direct + `site='none'` + `--any-rdkit`, python3.13 worker). Notebook never imports shim; only `ice_runner.py` subprocess does. rdkit will NOT be panel-exact 2025.03 — recorded deviation.
- Score: mean entropy similarity (query vs prediction) over covered spectra ([M+H]+/[M+Na]+, positive), `elucidation_over_candidates dist_func='entropy'` same caliber. Same z-fusion (z(base)+z(sim) equal weight, scored-first + unscored-by-base, top25 truncation) applied offline after retrieval — notebook only outputs raw scores.
- `enable_internet=true` DEV-only. Final submission stays offline. No competition submit in this notebook. Writes `iceberg_scores.json` + `iceberg_summary.json` to `/kaggle/working/`.
- Budget: `budget_s=2400` + grace. 250 mols, ~5499 cands trunc25 (30-mol precedent: 629 cands ~87s), est ~12-15 min + overhead.

In [ ]:
import sys, os, glob, json, time
print('python', sys.version)
print('INPUT_ROOT', sorted(os.listdir('/kaggle/input')))
ds_root = '/kaggle/input/datasets'
if os.path.isdir(ds_root):
    print('DATASETS_ROOT', sorted(os.listdir(ds_root)))
    for owner in sorted(os.listdir(ds_root)):
        op = os.path.join(ds_root, owner)
        if os.path.isdir(op):
            print(owner, sorted(os.listdir(op))[:10])
for pat in ['**/ice_runner.py', '**/np250_gl_input.json', '**/gen.pt', '**/inten.pt']:
    hits = sorted(glob.glob(f'/kaggle/input/{pat}', recursive=True))
    print(pat, '->', len(hits))
    for p in hits[:6]:
        print(' ', p)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print('mem_allocated_gb', round(torch.cuda.memory_allocated()/2**30, 3))
except Exception as e:
    print('torch probe failed', repr(e)[:500])
    raise


In [ ]:
import sys, os, glob, json, time, hashlib
t0 = time.time()
def find_first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(pattern, '->', len(hits))
    return hits
ICE_CANDS = find_first('/kaggle/input/**/ice_runner.py')
assert len(ICE_CANDS) >= 1, 'ice_runner.py not found under /kaggle/input: ' + str(sorted(os.listdir('/kaggle/input')))
ICE_PKG = os.path.dirname(ICE_CANDS[0])
print('ICE_PKG', ICE_PKG)
assert os.path.exists(os.path.join(ICE_PKG, 'fuse.py')), 'fuse.py missing'
# --- weight verification: msg_all mirror vs dataset MANIFEST (NIST excluded) ---
print('UPSTREAM msg_all: https://www.dropbox.com/scl/fo/kwm35ih8tlfnshfrcq8ot/AOeS4M0_v9MhqeEZys9sCRQ?rlkey=f1n6pbzx94g1k2el2wcbmee61&st=dcl9pbyp&dl=0')
print('UPSTREAM msg_simulation: https://www.dropbox.com/scl/fo/mcj0ngdvuj2xkhr983frb/ABGLrS8ZCeyYUWhzRfxPtRg?rlkey=rccvo52cehh75ma1yy8jji46t&st=4e3b946v&dl=0')
print('NIST weights: EXCLUDED (discussion/744470)')
manifest = json.load(open(os.path.join(ICE_PKG, 'MANIFEST.json')))
print('MANIFEST model:', manifest.get('model'), '| ms_pred_commit:', manifest.get('ms_pred_commit'), '| rdkit pin:', manifest.get('rdkit'))
for rel in ['ckpt/gen.pt', 'ckpt/inten.pt']:
    p = os.path.join(ICE_PKG, rel)
    assert os.path.exists(p), f'missing weight {rel}'
    exp = manifest['files'][rel]
    n = os.path.getsize(p)
    print(rel, 'bytes', n, 'expected', exp['bytes'], 'OK' if n == exp['bytes'] else 'SIZE_MISMATCH')
    assert n == exp['bytes'], f'size mismatch {rel}: {n} vs {exp["bytes"]}'
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for ch in iter(lambda: f.read(1 << 20), b''):
            h.update(ch)
    got = h.hexdigest()
    print(rel, 'sha256', got[:16] + '...', 'OK' if got == exp['sha256'] else 'HASH_MISMATCH')
    assert got == exp['sha256'], f'sha256 mismatch {rel}'
print('weights verified: msg_all mirror gen.pt + inten.pt, NIST excluded')
# --- NP250 input: np250_gl_input.json (= shortlists top50) truncated to top-25 ---
# Local check: np250_gl_input.json == np250_shortlists_top50.json top50, 0 mismatch;
# trunc25 = first 25, order preserved (same as iceberg-compare-30 precedent).
INP_CANDS = find_first('/kaggle/input/**/np250_gl_input.json')
assert len(INP_CANDS) >= 1, 'np250_gl_input.json not found'
INP = INP_CANDS[0]
print('INP', INP, round(os.path.getsize(INP) / 2**20, 2), 'MB')
full = json.load(open(INP))
assert isinstance(full, list) and len(full) == 250, f'expected 250 mols, got {len(full) if isinstance(full, list) else type(full)}'
items = []
for x in full:
    assert isinstance(x['mid'], str) and isinstance(x['cands'], list) and isinstance(x['spectra'], list), f'bad item {x.get("mid")}'
    assert len(x['cands']) >= 1 and len(x['spectra']) >= 1, f'empty item {x["mid"]}'
    for s in x['cands']:
        assert isinstance(s, str) and len(s) >= 2, f'bad SMILES in {x["mid"]}'
    cands25 = [str(s) for s in x['cands'][:25]]
    assert len(cands25) >= 1, f'empty trunc25 {x["mid"]}'
    items.append({'mid': str(x['mid']), 'cands': cands25, 'spectra': x['spectra']})
print('n_mols', len(items), 'n_cands_trunc25', sum(len(x['cands']) for x in items), 'n_cands_full', sum(len(x['cands']) for x in full), 'n_spectra', sum(len(x['spectra']) for x in items))
with open('/kaggle/working/ice_in_np250.json', 'w') as f:
    json.dump([{'mid': x['mid'], 'cands': x['cands'], 'n_spectra': len(x['spectra'])} for x in items], f)
sys.path.insert(0, ICE_PKG)
from fuse import run_ice
print(f'input ok in {time.time()-t0:.1f}s')


In [ ]:
import subprocess, sys
# cp312 wheels cannot install on this 3.13 worker -> PyPI fallback (DEV, internet on).
# rdkit will NOT be panel-exact 2025.03 -> --any-rdkit override; recorded as deviation.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
    'rdkit', 'einops', 'pytorch_lightning', 'lightning-utilities', 'torchmetrics'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-1500:])
import importlib
for m in ('rdkit', 'einops', 'pytorch_lightning', 'torchmetrics'):
    v = importlib.import_module(m)
    print(m, getattr(v, '__version__', 'ok'))


In [ ]:
import os, json, time, glob
t0 = time.time()
out = run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=2400.0, site='none', extra_args=('--any-rdkit',), log=print)
print(f'run_ice returned in {time.time()-t0:.1f}s, n_mols {len(out)}')
assert isinstance(out, dict) and len(out) == 250, f'ice output incomplete: {len(out) if isinstance(out, dict) else type(out)}'
with open('/kaggle/working/iceberg_scores.json', 'w') as f:
    json.dump(out, f)
print('wrote /kaggle/working/iceberg_scores.json', round(os.path.getsize('/kaggle/working/iceberg_scores.json')/1024, 1), 'KB')
import shutil
meta_src = '/kaggle/working/ice_work/ice_out.json.meta.json'
assert os.path.exists(meta_src), f'runner meta missing: {glob.glob("/kaggle/working/ice_work/*")}'
shutil.copyfile(meta_src, '/kaggle/working/ice_meta.json')
meta = json.load(open(meta_src))
print(json.dumps({k: meta.get(k) for k in ('status','n_mols','n_mols_covered','n_mols_scored','n_cands','n_scored','seconds','pred_per_s','device','rdkit','max_mem_gb')}, indent=1))


In [ ]:
import json, os
out = json.load(open('/kaggle/working/iceberg_scores.json'))
meta = json.load(open('/kaggle/working/ice_meta.json'))
n_cands = sum(len(v) for v in out.values())
n_scored = sum(1 for v in out.values() for s in v.values() if s is not None)
n_null = n_cands - n_scored
per_mol = {mid: {'n_cands': len(v), 'n_scored': sum(1 for s in v.values() if s is not None)} for mid, v in out.items()}
n_mols_scored = sum(1 for v in per_mol.values() if v['n_scored'] > 0)
n_mols_null = len(per_mol) - n_mols_scored
secs = float(meta.get('seconds', 0) or 0)
summary = {
  'n_molecules': len(out),
  'n_cands': int(n_cands),
  'n_scored': int(n_scored),
  'n_null': int(n_null),
  'n_mols_scored': int(n_mols_scored),
  'n_mols_all_null': int(n_mols_null),
  'seconds': secs,
  'sec_per_cand': (secs/max(1,n_cands)),
  'sec_per_scored': (secs/max(1,n_scored)),
  'max_mem_gb': meta.get('max_mem_gb'),
  'device': meta.get('device'),
  'rdkit': meta.get('rdkit'),
  'status': meta.get('status'),
  'pred_per_s': meta.get('pred_per_s'),
  'per_molecule': per_mol,
  'runner_meta': meta,
}
print(json.dumps({k: summary[k] for k in ('n_molecules','n_cands','n_scored','n_null','n_mols_scored','n_mols_all_null','seconds','sec_per_cand','sec_per_scored','max_mem_gb','device','rdkit','status','pred_per_s')}, indent=1))
with open('/kaggle/working/iceberg_summary.json', 'w') as f:
    json.dump(summary, f, indent=1)
print('wrote /kaggle/working/iceberg_summary.json + iceberg_scores.json; list:', os.listdir('/kaggle/working'))
